In [1]:
from pathlib import Path
DOCS = Path('~/pytorch/docs/source')

In [2]:
files = list(DOCS.rglob("*.md"))
print(len(files))
print(files[:5])

256
[PosixPath('~/pytorch/docs/source/func.batch_norm.md'), PosixPath('~/pytorch/docs/source/nn.attention.bias.md'), PosixPath('~/pytorch/docs/source/storage.md'), PosixPath('~/pytorch/docs/source/fx.experimental.md'), PosixPath('~/pytorch/docs/source/threading_environment_variables.md')]


In [9]:
text = files[2].read_text(encoding="utf-8")
print(len(text))

8579


In [11]:
import re
text = re.sub(r'```{eval-rst}.*?```', "", text, flags=re.DOTALL)
print(len(text))
print("autosummary" in text, "{eval-rst}" in text)

7206
False False


In [16]:
from collections import Counter

ctr = Counter()
for file in files:
    text_file = file.read_text(encoding="utf-8")
    ctr.update(re.findall(r"`{3,}\{([\w-]+)\}", text))

print(ctr.most_common())

[('eval-rst', 1749), ('code-cell', 114), ('note', 74), ('code-block', 63), ('warning', 52), ('toctree', 46), ('image', 33), ('math', 28), ('contents', 11), ('figure', 9), ('currentmodule', 8), ('seealso', 7), ('list-table', 5), ('raw', 4), ('role', 3), ('testcode', 2), ('testoutput', 2), ('csv-table', 2), ('tip', 2), ('dropdown', 1), ('Note', 1), ('mermaid', 1)]


In [21]:
ctr1 = Counter()
for file in files:
    text = file.read_text(encoding="utf-8")
    blocks = re.findall(r"`{3,}\{eval-rst\}(.*?)`{3,}", text, flags=re.DOTALL)
    for block in blocks:
        names = re.findall(r"\.\. ([\w:-]+)::", block, flags=re.DOTALL)
        ctr1.update(names)
print(ctr1.most_common())

[('autofunction', 604), ('py:module', 588), ('autoclass', 407), ('currentmodule', 366), ('autosummary', 317), ('automodule', 248), ('literalinclude', 54), ('automethod', 34), ('role', 28), ('py:function', 22), ('attribute', 21), ('toctree', 10), ('class', 10), ('list-table', 8), ('autoattribute', 6), ('csv-table', 5), ('note', 5), ('math', 4), ('code-block', 4), ('warning', 4), ('deprecated', 3), ('method', 2), ('grid-item-card', 2), ('image', 1), ('autoexception', 1), ('autodata', 1), ('seealso', 1), ('grid', 1)]


In [24]:
from collections import defaultdict

examples = defaultdict(list)
targets = ["class", "attribute", "method", "py:function", "literalinclude"]

for file in files:
    text = file.read_text(encoding="utf-8")
    blocks = re.findall(r"`{3,}\{eval-rst\}(.*?)`{3,}", text, flags=re.DOTALL)
    for block in blocks:
        for name in set(re.findall(r"\.\. ([\w:-]+)::", block)):
            if name in targets and len(examples[name]) < 2 and block not in [b for _, b in examples[name]]:
                examples[name].append((file.name, block))

for name, items in examples.items():
    print("=" * 20, name)
    for fname, block in items:
        print("--", fname)
        print(block)

==================== attribute
-- backends.md

.. attribute::  fp32_precision

    A :class:`str` that controls the internal computation precision of CUDA
    matrix multiplications with FP32 inputs and outputs. Supported values are
    ``"ieee"``, ``"tf32"``, and ``"bfx9"``. The ``"bfx9"`` value allows
    cuBLAS BF16x9 and requires a PyTorch build with CUDA 12.9 or newer. BF16x9
    emulation is available on NVIDIA GPUs with compute capability 10.0 or 10.3;
    cuBLAS uses native FP32 on other architectures. See :ref:`tf32_on_ampere`.

-- backends.md

.. attribute::  allow_tf32

    A :class:`bool` that controls whether TensorFloat-32 tensor cores may be used in matrix
    multiplications on Ampere or newer GPUs. allow_tf32 is going to be deprecated. See :ref:`tf32_on_ampere`.

==================== method
-- backends.md

.. attribute::  cufft_plan_cache

    ``cufft_plan_cache`` contains the cuFFT plan caches for each CUDA device.
    Query a specific device `i`'s cache via `torch.ba

In [23]:
n = 0
for file in files:
    text = file.read_text(encoding="utf-8")
    found = re.findall(r"^`{4,}.*$", text, flags=re.M)
    if found:
        n += len(found)
        print(file.name, found[:3])
print(n)

tensor_attributes.md ['````{note}', '````', '````{note}']
6


In [26]:
import textwrap

def split_directives(block):
    result = []
    header = None
    body = []
    for line in block.splitlines():
        if line.startswith('.. '):
            if header:
                result.append((header, textwrap.dedent("\n".join(body)).strip()))

            header = line
            body = []
        else:
            body.append(line)
    if header:
        result.append((header, textwrap.dedent("\n".join(body)).strip()))
    return result

# 1. один блок из multiprocessing.md
for h, b in split_directives(examples["class"][1][1]):
    print(repr(h))
    print(b)

# 2. число пар должно совпасть с числом строк ".. py:function::"
block = examples["py:function"][1][1]
pairs = split_directives(block)
print(len(pairs), block.count("\n.. py:function::") + block.startswith(".. py:function::"))

'.. class:: SpawnContext'
Returned by :func:`~spawn` when called with ``join=False``.

.. automethod:: join
17 17


In [30]:
def parse_header(header):
    m = re.match(r"\.\. ([\w:-]+)::(?:\s+([\w.]+))?", header)
    if m:
        return (m.group(1), m.group(2))
    else:
        return None

bad = []
for file in files:
    text = file.read_text(encoding="utf-8")
    for block in re.findall(r"`{3,}\{eval-rst\}(.*?)`{3,}", text, flags=re.DOTALL):
        for header, body in split_directives(block):
            if parse_header(header) is None:
                bad.append(header)
print(len(bad))
print(bad[:5])

6
['.. This module needs to be documented. Adding here in the meantime', '.. for tracking purposes', '.. _Link 1: https://arxiv.org/abs/1611.00712', '.. _Link 2: https://arxiv.org/abs/1611.01144', '.. These modules are missing docs. Adding them here only for tracking']


In [31]:
print(parse_header(".. autosummary::"))               # ('autosummary', None)
print(parse_header(".. attribute::  fp32_precision"))

('autosummary', None)
('attribute', 'fp32_precision')


In [34]:
sm = [f for f in files if f.name == "symmetric_memory.md"]
print(sm)

[PosixPath('~/pytorch/docs/source/distributed/symmetric_memory.md')]


In [121]:
POINTERS = {"autofunction", "autoclass", "automethod", "autoattribute",
            "autodata", "autoexception"}
SERVICE = {"toctree", "role", "image", "raw", "contents"}
CONTENT_KINDS = {"note", "warning", "seealso", "deprecated", "hint", "math"}

def full_name(module, name):
    if module is None:
        return name
    if not name.startswith("torch."):
        name =  module + "." + name
    return name
def process_file(path):
    text = path.read_text(encoding="utf-8")
    current_module = None
    chunks, pointers = [], []
    for block in re.findall(r"`{3,}\{eval-rst\}(.*?)`{3,}", text, flags=re.DOTALL):
        for header, body in split_directives(block):
            parsed = parse_header(header)
            if parsed is None:
                continue
            kind, name = parsed
            if kind in ("currentmodule", "automodule", "py:module"):
                current_module = name
            elif kind in POINTERS:
                pointers.append(full_name(current_module, name))
            elif kind == "autosummary":
                for line in body.splitlines():
                    line = line.strip()
                    if line.strip() and not line.strip().startswith(":"):
                        pointers.append(full_name(current_module, line))
            elif kind in SERVICE:
                continue
            else:
                if kind in CONTENT_KINDS:
                    rest = header.split("::", 1)[1].strip()
                    text_ = (rest + "\n" + body).strip()
                    chunks.append({"module": current_module, "name": None, "kind": kind, "text": text_})
                else:
                    if name:
                        pointers.append(full_name(current_module, name))
                    chunks.append({"module": current_module, "name": name, "kind": kind, "text": body})
    return chunks, pointers


chunks, pointers = process_file(DOCS / "accelerator.md")
print(len(chunks), pointers[:3])
chunks, pointers = process_file(DOCS / "distributed/symmetric_memory.md")
print(len(chunks)); print(chunks[0])

0 ['torch.accelerator.device_count', 'torch.accelerator.is_available', 'torch.accelerator.current_accelerator']
22
{'module': 'torch.ops.symm_mem', 'name': 'fused_all_gather_matmul', 'kind': 'py:function', 'text': 'Overlaps an all-gather of ``A_shard`` with one or more matrix\nmultiplications. It is semantically equivalent to::\n\n    A = all_gather_single(A_shard, gather_dim, group)\n    outputs = [torch.matmul(A, B) for B in Bs]\n\nThe tensors in ``Bs`` must be 2-D and compatible with ``A`` for matrix\nmultiplication. ``A_shard`` may be an ordinary accelerator tensor; the\noperator allocates and reuses symmetric workspace internally. All ranks\nmust call the operator with matching shapes and arguments.\n\nFor best performance, arrange ``A_shard`` so that\n``A_shard.movedim(gather_dim, 0)`` is contiguous. Set ``return_A=False``\nwhen only the matrix-multiplication results are needed; this can avoid\nmaterializing the gathered tensor on supported systems. Run the operator\nonce before 

In [39]:
def split_sections(text):
    sections = []
    stack, lines, fence = [], [], 0
    for line in text.splitlines():
        s = line.strip()
        m = re.match(r"^(`{3,})", s)
        if m:
            if fence == 0:
                fence = len(m.group(1))
            elif re.fullmatch(r"`{3,}", s) and len(s) >= fence:
                fence = 0
            lines.append(line)
            continue
        if fence == 0:
            h = re.match(r"^(#{1,6})\s+(.*)$", line)
            if h:
                level = len(h.group(1))
                title = h.group(2)
                body = "\n".join(lines).strip()
                if body:
                    sections.append({"path": list(stack), "text": body})
                stack = stack[:level-1] + [title]
                lines = []
                continue
        lines.append(line)
    body = "\n".join(lines).strip()
    if body:
        sections.append({"path": list(stack), "text": body})
    return sections

In [40]:
test = """# A
text a
```python
# not a heading
```
## B
text b
"""
for s in split_sections(test):
    print(s["path"], repr(s["text"]))

['A'] 'text a\n```python\n# not a heading\n```'
['A', 'B'] 'text b'


In [41]:
test2 = """# A
````{note}
```python
# inner comment
```
text inside note
````
## B
text b
"""
for s in split_sections(test2):
    print(s["path"], repr(s["text"]))

['A'] '````{note}\n```python\n# inner comment\n```\ntext inside note\n````'
['A', 'B'] 'text b'


In [44]:
all_sections = []
for f in files:
    for sec in split_sections(f.read_text(encoding="utf-8")):
        all_sections.append((f.name, sec))
lens = sorted(len(sec["text"]) for _, sec in all_sections)
print(len(all_sections), lens[len(lens)//2], lens[-5:])

big = sorted(all_sections, key=lambda x: -len(x[1]["text"]))[:5]
for name, sec in big:
    print(name, sec["path"], len(sec["text"]), "eval-rst" in sec["text"])

2016 587 [9896, 10568, 11683, 12484, 21463]
symmetric_memory.md ['PyTorch Symmetric Memory', 'Op Reference'] 21463 True
tensors.md ['torch.Tensor', 'Tensor class reference'] 12484 True
cuda.md ['CUDA semantics', 'Memory management', 'Optimizing memory usage  with `PYTORCH_ALLOC_CONF`'] 11683 False
checkpoint.md ['Distributed Checkpoint - torch.distributed.checkpoint', 'Additional resources:'] 10568 True
distributed.md ['Distributed communication package - torch.distributed', 'Read More'] 9896 True


In [122]:
def replace_open(m):
    kind, arg = m.group(1), m.group(2)
    if kind in ("note", "hint", "warning"):
        return kind.capitalize()+":"
    if kind == "tab-item":
        return "Tab: "+arg
    return ""


def clean_colon_blocks(text):
    text = re.sub(r"^:{3,}\{([\w-]+)\}[ \t]*(.*)$", replace_open, text, flags=re.M)
    text = re.sub(r"^:{3,}[ \t]*$", "", text, flags=re.M)
    return text
def clean_misc(text):
    text = re.sub(r"^%.*$", "", text, flags=re.M)                      
    text = re.sub(r"^\([^)\n]+\)=[ \t]*$", "", text, flags=re.M)       
    text = re.sub(r"<!--.*?-->", "", text, flags=re.DOTALL)
    text = re.sub(r"^`{3,}\{role\}.*?^`{3,}[ \t]*$", "", text, flags=re.M | re.DOTALL)
    text = re.sub(r"^[ \t]*:class:.*$", "", text, flags=re.M)
    return text

def clean_roles(text):
    text = re.sub(r"\{[\w:-]+\}(`[^`]+`)", r"\1", text)
    text = re.sub(r"`~", "`", text)
    return text

EVAL_RST = r"`{3,}\{eval-rst\}.*?`{3,}"

def load_file(path):
    text = path.read_text(encoding="utf-8")
    api_chunks, pointers = process_file(path)
    clean = re.sub(EVAL_RST, "", text, flags=re.DOTALL)
    clean = clean_colon_blocks(clean)
    clean = clean_misc(clean)
    clean = clean_roles(clean)
    clean = clean_rst(clean)
    sections = split_sections(clean)
    return sections, api_chunks, pointers

ext = [f for f in files if f.name == "extending.md"]
print(ext)
t = ext[0].read_text(encoding="utf-8")
c = clean_colon_blocks(t)
print(len(re.findall(r"^:{3,}", t, flags=re.M)), len(re.findall(r"^:{3,}", c, flags=re.M)))
m = re.search(r"^Note:$", c, flags=re.M)
print(m)
if m:
    print(c[m.start()-100:m.start()+400])

[PosixPath('~/pytorch/docs/source/notes/extending.md')]
6 0
<re.Match object; span=(19298, 19303), match='Note:'>
t the output is
    def my_cube(x):
        result, dx = MyCube.apply(x)
        return result

```
Note:
Inputs to ``backward``, i.e., {attr}`grad_output`, can also be tensors that
track history. So if ``backward`` is implemented with differentiable
operations, (e.g., invocation of another custom
{class}`~torch.autograd.Function`), higher order derivatives will work.
In this case, the tensors saved with ``save_for_backward`` can also be used
in the backward and have gradients flowing back but t


In [53]:
secs = []
for file in files:
    sections, api_chunks, pointers = load_file(file)
    for sec in sections:
        secs.append((file.name, sec))

lens = sorted(len(sec["text"]) for _, sec in secs)
print(len(secs), lens[len(lens)//2], lens[-5:])
print(sum(1 for _, s in secs if len(s["text"]) > 2000))

1644 661 [8618, 8913, 9286, 9732, 11683]
222


In [48]:
ctr2 = Counter()
for f in files:
    ctr2.update(re.findall(r"^:{3,}\{([\w-]+)\}", f.read_text(encoding="utf-8"), flags=re.M))
print(ctr2.most_common())

[('note', 45), ('warning', 38), ('tab-set', 5), ('tab-item', 5), ('tab-set-code', 3), ('figure', 2), ('list-table', 1), ('role', 1), ('currentmodule', 1), ('hint', 1)]


In [65]:
def split_units(text):
    units, cur, fence = [], [], 0
    for line in text.splitlines():
        s = line.strip()
        m = re.match(r"^(`{3,})", s)
        if m:
            if fence == 0:
                fence = len(m.group(1))
            elif re.fullmatch(r"`{3,}", s) and len(s) >= fence:
                fence = 0
            cur.append(line)
            continue
        if fence == 0 and not s:
            body = "\n".join(cur).strip()
            if body:
                units.append(body)
            cur = []
            continue
        if fence == 0 and re.match(r"^([*-]|\d+[.)])\s", line) and cur:
            body = "\n".join(cur).strip()
            if body:
                units.append(body)
            cur = []
        cur.append(line)
    body = "\n".join(cur).strip()
    if body:
        units.append(body)
    return units


u = split_units(sec["text"])
print(len(u), max(len(x) for x in u))
print(all(x.count("```") % 2 == 0 for x in u))

25 1944
True


In [62]:
def split_long(text, max_chars=1500):
    chunks, cur = [], ""
    for unit in split_units(text):
        if not cur:
            cur = unit
        elif len(cur) + 2 + len(unit) <= max_chars:
            cur = cur + "\n\n" + unit
        else:
            chunks.append(cur)
            cur = unit
    if cur:
        chunks.append(cur)
    return chunks

all_parts = []
for name, s in secs:
    for p in split_long(s["text"]):
        all_parts.append((name, s["path"], p))
plens = sorted(len(p) for _, _, p in all_parts)
print(len(all_parts), plens[len(plens)//2], plens[-5:])
print(sum(1 for x in plens if x < 100))
big = sorted(all_parts, key=lambda x: -len(x[2]))[:5]
for name, path, p in big:
    print(name, path, len(p), p.count("```"))
    print(p[:150].replace("\n", " "))
    print("---")
short = [x for x in all_parts if len(x[2]) < 100]
for name, path, p in short[:8]:
    print(name, path, repr(p))

2211 713 [3266, 3357, 3579, 4710, 8249]
189
torch_nccl_environment_variables.md ['PYTORCH NCCL Backend Environment Variables'] 8249 2
```{list-table} :header-rows: 1  * - **Variable**   - **Description** * - ``TORCH_DIST_NAN_CHECK``   - **All backends.** If set to ``1``, attach a bac
---
extending.func.md ['Extending torch.func with autograd.Function', '{func}`torch.vmap` Support', 'Defining the vmap staticmethod'] 4710 2
```python def to_numpy(tensor):     return tensor.cpu().numpy()  class NumpySort(torch.autograd.Function):     @staticmethod     def forward(x, dim): 
---
cuda.md ['CUDA semantics', 'Tuning NVLink Performance with Custom Memory Allocator on H100/H200 GPUs'] 3579 2
```cpp // assuming a system with 8 GPUs struct CustomAllocInfo {   void** devPtr;  // This will be the usable virtual memory address   CUdeviceptr dpt
---
rpc.md ['Distributed RPC Framework', 'Basics'] 3357 0
1) **Remote Procedure Call (RPC)** supports running a function on the specified    destination worke

In [70]:
all_parts = []
for file in files:
    sections, _, _ = load_file(file)
    for s in sections:
        for p in split_long(s["text"]):
            all_parts.append((file.name, s["path"], p))
plens = sorted(len(p) for _, _, p in all_parts)
print(len(all_parts), plens[len(plens)//2], plens[-5:])
print(sum(1 for x in plens if x < 100))

print(sum("{ref}" in p or "{class}" in p or "{meth}" in p for _, _, p in all_parts))

2098 751 [3225, 3266, 3579, 4710, 8248]
92
0


In [67]:
short = [x for x in all_parts if len(x[2]) < 100]
for name, path, p in short[:10]:
    print(name, path, repr(p))

fx.experimental.md ['torch.fx.experimental'] 'Warning:\nThese APIs are experimental and subject to change without notice.'
mtia.md ['torch.mtia'] 'The MTIA backend is implemented out of the tree, only interfaces are defined\nhere.'
torch.compiler_api.md ['torch.compiler API reference'] 'For a quick overview of `torch.compiler`, see {ref}`torch.compiler_overview`.'
utils.md ['torch.utils.hipify.hipify_python'] '<!-- This module needs to be documented. Adding here in the meantime\nfor tracking purposes -->'
autograd.md ['Automatic differentiation package - torch.autograd', 'Context method mixins'] 'When creating a new {class}`Function`, the following members are available to `ctx`.'
autograd.md ['Automatic differentiation package - torch.autograd', 'Custom Function utilities'] 'Decorator for backward method.\n\nBase custom {class}`Function` used to build PyTorch utilities'
linalg.md ['torch.linalg', 'Decompositions'] '(linalg solvers)='
linalg.md ['torch.linalg', 'Solvers'] '(linalg inve

In [123]:
def clean_rst(text):
    text = re.sub(r":[\w:-]+:(`[^`]+`)", r"\1", text)
    text = re.sub(r"`~", "`", text)
    text = re.sub(r"``([^`]+)``", r"`\1`", text)
    return text


def build_chunks(files):
    result, seen = [], set()
    for file in files:
        sections, api_chunks, pointers = load_file(file)
        rel = str(file.relative_to(DOCS))
        for s in sections:
            title = " > ".join(s["path"])
            for p in split_long(s["text"]):
                if len(p.strip()) < 20:
                    continue
                text = title + "\n" + p
                if text in seen:
                    continue
                seen.add(text)
                result.append({"id":len(result), "file":rel, "kind": "text", "title": title, "text": text})
        for c in api_chunks:
            label = c["name"] or c["kind"]
            if c["module"]:
                title = c["module"] + "." + label
            else:
                title = label
            body = clean_rst(c["text"])
            if len(body.strip()) < 20:
                continue
            if c["kind"] in ("note", "warning", "seealso"):
                body = c["kind"].capitalize() + ":\n" + body
            text = title + "\n" + body
            if text in seen:
                continue
            seen.add(text)
            result.append({"id":len(result), "file":rel, "kind": "api", "title": title, "text": text})
    return result

chunks_all = build_chunks(files)
print(len(chunks_all), sum(c["kind"] == "api" for c in chunks_all))
api = next(c for c in chunks_all if c["title"].endswith("fp32_precision"))
print(api["text"])
print(sum(":class:" in c["text"] or ":ref:" in c["text"] or ":func:" in c["text"] for c in chunks_all))

dropped = Counter()
for file in files:
    for c in load_file(file)[1]:
        if not c["name"]:
            dropped[c["kind"]] += 1
print(dropped.most_common())

2350 126
torch.backends.cuda.matmul.fp32_precision
A `str` that controls the internal computation precision of CUDA
matrix multiplications with FP32 inputs and outputs. Supported values are
`"ieee"`, `"tf32"`, and `"bfx9"`. The `"bfx9"` value allows
cuBLAS BF16x9 and requires a PyTorch build with CUDA 12.9 or newer. BF16x9
emulation is available on NVIDIA GPUs with compute capability 10.0 or 10.3;
cuBLAS uses native FP32 on other architectures. See `tf32_on_ampere`.
1
[('list-table', 7), ('note', 5), ('csv-table', 4), ('warning', 4), ('math', 2), ('seealso', 1), ('deprecated', 1)]


In [124]:
left = [c for c in chunks_all if ":class:" in c["text"] or ":ref:" in c["text"] or ":func:" in c["text"]]
for c in left:
    t = c["text"]
    i = max(t.find(":class:"), t.find(":ref:"), t.find(":func:"))
    print(c["title"], repr(t[max(0, i-30):i+50]))

Advanced Options to Control Dynamic Behavior > Profile-Guided Optimization (PGO) > Identifying Dynamic Elements Marked by PGO "users/bobren/a/pytorch/r2.py:2:func:\n   L['m']: fully dynamic scalar or tensor\n "


In [125]:
s = "before\n```{role} hidden\n:class: hidden-section\n```\nafter"
print(repr(clean_misc(s)))   # ожидание: 'before\n\nafter'

chunks_all = build_chunks(files)
left = [c for c in chunks_all if ":class:" in c["text"] or ":ref:" in c["text"] or ":func:" in c["text"]]
print(len(chunks_all), len(left))

print(Counter(c["kind"] for c in chunks_all))

for c in left:
    t = c["text"]
    i = max(t.find(":class:"), t.find(":ref:"), t.find(":func:"))
    print(c["title"], repr(t[max(0, i-30):i+50]))

'before\n\nafter'
2350 1
Counter({'text': 2224, 'api': 126})
Advanced Options to Control Dynamic Behavior > Profile-Guided Optimization (PGO) > Identifying Dynamic Elements Marked by PGO "users/bobren/a/pytorch/r2.py:2:func:\n   L['m']: fully dynamic scalar or tensor\n "


In [126]:
print(Counter(c["kind"] for c in chunks_all))
for c in chunks_all:
    if c["kind"] == "api" and c["title"].split(".")[-1] in ("note", "warning", "list-table"):
        print(c["title"], repr(c["text"][:120]))
        break

Counter({'text': 2224, 'api': 126})
torch.utils.model_dump.warning 'torch.utils.model_dump.warning\nWarning:\n`torch.utils.model_dump` is deprecated and will be removed in a future\nrelease. '


In [127]:
for c in chunks_all:
    if "hidden-section" in c["text"]:
        print(c["id"], c["file"])

for file in files:
    for c in load_file(file)[1]:
        body = clean_rst(c["text"]).strip()
        if len(body) < 20:
            print(file.name, c["kind"], c["name"], repr(body))

type_info.md class torch.finfo ''
type_info.md class torch.iinfo ''
tensor_attributes.md class dtype ''
tensor_attributes.md class device ''
tensor_attributes.md class layout ''
tensor_attributes.md class memory_format ''


In [128]:
chunks_all = build_chunks(files)
print(len(chunks_all))
for c in chunks_all:
    if "hidden-section" in c["text"]:
        print(c["id"], c["file"], repr(c["text"]))

2350


In [129]:
print(repr(clean_misc("\n:class: hidden-section")))

'\n'


In [130]:
f = [f for f in files if f.name == "tensor.parallel.md"][0]
t = f.read_text(encoding="utf-8")
i = t.find("hidden-section")
print(repr(t[max(0, i-200):i+100]))

':::{role} hidden\n    :class: hidden-section\n:::\n\n# Tensor Parallelism - torch.distributed.tensor.parallel\n\nTensor Parallelism(TP)'


In [133]:
f = [f for f in files if f.name == "library.md"][0]
for c in load_file(f)[1]:
    if c["kind"] == "warning":
        print(repr(c["name"]), repr(c["text"][:200]))

None 'The low-level operator registration APIs and the PyTorch Dispatcher are a complicated PyTorch concept. We recommend you use the higher level APIs above (that do not require a torch.library.Library obj'


In [142]:
import json
chunks_all = build_chunks(files)
print(len(chunks_all), Counter(c["kind"] for c in chunks_all))
with open("../data/chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks_all, f, ensure_ascii=False, indent=1)

# 1. уже есть: сбор указателей
all_pointers = set()
for file in files:
    all_pointers.update(load_file(file)[2])

# 2. новая: фильтр
valid = re.compile(r"^torch(\.\w+)*$")
names = sorted(n for n in all_pointers if valid.match(n))
print(len(all_pointers), len(names))

rejected = [n for n in all_pointers if not valid.match(n)]
print(rejected)

2350 Counter({'text': 2224, 'api': 126})
4314 4305
['torch.nn.~parameter.UninitializedParameter', 'torch.nn.~parameter.Parameter', 'TorchDynamo', 'torch.nn.~parameter.Buffer', '2', '..', 'torch.nn.~parameter.UninitializedBuffer', 'yaml', 'Title']


In [136]:
import random
lst = sorted(all_pointers)
print(lst[:15])
print(random.sample(lst, 15))
print([n for n in lst if " " in n or not n.startswith("torch")][:15])

['..', '2', 'Title', 'TorchDynamo', 'torch.BFloat16Storage', 'torch.BoolStorage', 'torch.ByteStorage', 'torch.CharStorage', 'torch.ComplexDoubleStorage', 'torch.ComplexFloatStorage', 'torch.DoubleStorage', 'torch.Event', 'torch.FloatStorage', 'torch.Generator', 'torch.HalfStorage']
['torch.ao.quantization.quantize_fx.convert_to_reference_fx', 'torch.fx.passes.splitter_base.generate_inputs_for_submodules', 'torch.fx.experimental.unification.multipledispatch.dispatcher.variadic_signature_matches', 'torch.nn.functional.celu', 'torch.fx.passes.infra.pass_manager.this_before_that_pass_constraint', 'torch.Tensor.remainder', 'torch.rand', 'torch.unfold_copy', 'torch.utils.cpp_extension.CppExtension', 'torch.foreach.abs', 'torch.get_device', 'torch.fx.experimental.migrate_gradual_types.transform_to_z3.transform_to_z3', 'torch.ao.nn.intrinsic.ConvReLU3d', 'torch.nested.as_nested_tensor', 'torch.backends.nnpack.flags']
['..', '2', 'Title', 'TorchDynamo', 'yaml']


In [139]:
%pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.6/138.6 MB 3.8 MB/s  0:00:36 eta 0:00:010:00:02
Note: you may need to restart the kernel to use updated packages.


In [140]:
import importlib, inspect
import torch

def get_doc(full):
    parts = full.split(".")
    for k in range(len(parts), 0, -1):
        try:
            obj = importlib.import_module(".".join(parts[:k]))
        except Exception:
            continue
        try:
            for attr in parts[k:]:
                obj = getattr(obj, attr)
            return inspect.getdoc(obj)
        except Exception:
            return None
    return None

print(get_doc("torch.accelerator.device_count"))
print(get_doc("torch.nn.Linear")[:200])
print(get_doc("torch.Tensor.add"))

sample = random.sample(lst, 100)
ok = [n for n in sample if get_doc(n)]
print(len(ok))
print([n for n in sample if not get_doc(n)][:15])

Return the number of current :ref:`accelerator<accelerators>` available.

Returns:
    int: the number of the current :ref:`accelerator<accelerators>` available.
        If there are no available accelerators, return 0.

.. note:: This API delegates to the device-specific version of `device_count`.
    On CUDA, this API will NOT poison fork if NVML discovery succeeds.
    Otherwise, it will. For more details, see :ref:`multiprocessing-poison-fork-note`.
Applies an affine linear transformation to the incoming data: :math:`y = xA^T + b`.

This module supports :ref:`TensorFloat32<tf32_on_ampere>`.

On certain ROCm devices, when using float16 inputs this
add(other, *, alpha=1) -> Tensor

Add a scalar or tensor to :attr:`self` tensor. If both :attr:`alpha`
and :attr:`other` are specified, each element of :attr:`other` is scaled by
:attr:`alpha` before being used.

When :attr:`other` is a tensor, the shape of :attr:`other` must be
:ref:`broadcastable <broadcasting-semantics>` with the shape 

W1101 23:01:09.686000 75825 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


87
['torch.cudnn_convolution_transpose', 'torch.distributed.functional_collectives.all_reduce_coalesced', 'torch.fx.experimental.unification.multipledispatch.dispatcher.variadic_signature_matches', 'torch.ao.quantization.utils.get_qparam_dict', 'torch.fx.experimental.sym_node.is_channels_last_contiguous_3d', 'torch.distributed.functional_collectives.all_reduce', 'torch.fx.experimental.proxy_tensor.proxy_call', 'torch.ao.quantization.backend_config.utils.get_fuser_method_mapping', 'torch.distributed.distributed_c10d.FlightRecorderHook', 'torch.backends.cuda.SDPAParams', 'torch.dtype', 'TorchDynamo', 'torch.fx.experimental.migrate_gradual_types.constraint_generator.maxpool_inference_rule']
